## Setup

Initialize the notebook environment:
1. Add project root to Python path
2. Configure logging with deduplication (same as backend)
3. **Disable Application Insights** to prevent polluting production telemetry
4. **Disable blob storage** to prevent polluting production data

In [ ]:
import os
import sys

# Add project root to path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

In [ ]:
# Setup logging using the project's logging configuration
# This ensures consistent logging behavior with the main application
from backend.tools.logger import setup_logging

# Initialize logging with deduplication (same as backend)
# IMPORTANT: Disable Application Insights for local testing to avoid polluting production data
log_file = setup_logging(log_prefix="notebook", enable_app_insights=False)
print(f"Logging initialized: {log_file}")
print("✓ Application Insights DISABLED for local testing")

# Get a logger for this notebook
import logging
logger = logging.getLogger("notebook")
logger.info("Notebook logger initialized successfully")

## Local Testing Mode

The following cell disables blob storage to prevent polluting production data. 
All data will be stored locally in `~/.adalflow/` during notebook testing.

In [ ]:
# IMPORTANT: Disable blob storage for local notebook testing
# This prevents accidentally polluting production blob data during development
import backend.blob_storage as blob_storage

# Store original function reference
_original_is_blob_storage_configured = blob_storage.is_blob_storage_configured

# Override to always return False in notebook context
def _disabled_blob_storage():
    """Always return False to force local storage mode during notebook testing."""
    return False

blob_storage.is_blob_storage_configured = _disabled_blob_storage

# Also ensure any cached blob client is cleared
blob_storage._blob_client = None

print("✓ Blob storage DISABLED for local testing")
print("  All data will use local ~/.adalflow/ storage")
print("  Production blob data will NOT be affected")

In [ ]:
# Verify blob storage is disabled
from backend.blob_storage import is_blob_storage_configured
print(f"Blob storage configured: {is_blob_storage_configured()}")
assert not is_blob_storage_configured(), "Blob storage should be disabled for local testing!"
print("✓ Verified: Blob storage is disabled")

## Configuration Check

Verify Azure OpenAI configuration is properly set up.

In [ ]:
from backend.config import get_model_config, is_azure_openai_configured

# Check if Azure OpenAI is configured
print(f"Azure OpenAI configured: {is_azure_openai_configured()}")

# Get model configuration
config = get_model_config()
print(f"\nModel config:")
print(f"  Provider: {config.get('provider', 'N/A')}")
print(f"  Model: {config.get('model', 'N/A')}")

## Test Azure OpenAI Connection

Test the connection to Azure OpenAI with a simple completion.

In [ ]:
from backend.azureai_client import AzureAIClient
from backend.config import get_model_config, get_azure_deployment_name, get_azure_openai_config
from adalflow.core.types import ModelType

# Show Azure OpenAI config for debugging
azure_config = get_azure_openai_config()
print(f"Azure OpenAI Config:")
print(f"  Endpoint: {azure_config.get('endpoint')}")
print(f"  Deployment: {azure_config.get('deployment')}")
print(f"  API Version: {azure_config.get('api_version')}")

# Initialize client
config = get_model_config()
client = AzureAIClient(**config.get('initialize_kwargs', {}))

# Test simple completion
deployment = get_azure_deployment_name()
print(f"\nUsing deployment: {deployment}")

try:
    response = client.call(
        api_kwargs={
            "messages": [{"role": "user", "content": "Say hello in one word."}],
            "model": deployment,
            "temperature": 0.7,
            "max_tokens": 10
        },
        model_type=ModelType.LLM
    )
    print(f"Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

## Test Embeddings

Test the embedding generation with Azure OpenAI.

In [ ]:
from backend.config import get_embedder_config
from backend.tools.embedder import get_embedder

# Get embedder configuration
embedder_config = get_embedder_config()
print(f"Embedder config:")
print(f"  Provider: {embedder_config.get('provider', 'N/A')}")
print(f"  Model: {embedder_config.get('model', 'N/A')}")

# Initialize embedder using the factory function
embedder = get_embedder()

# Test embedding
test_text = "Hello, world!"
result = embedder(test_text)  # Returns EmbedderOutput

# Extract the embedding data from EmbedderOutput
embedding = result.data[0].embedding  # Access first embedding's data
print(f"\nEmbedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

## 5. Repository Input

Enter the repository URL and optionally provide a Personal Access Token (PAT) for private repositories.

In [ ]:
# ============================================
# USER INPUT: Configure your repository here
# ============================================

# Repository URL (required)
# Examples:
#   - GitHub: "https://github.com/owner/repo"
#   - GitLab: "https://gitlab.com/owner/repo"
#   - Azure DevOps: "https://dev.azure.com/org/project/_git/repo"
#   - Bitbucket: "https://bitbucket.org/owner/repo"

REPO_URL = "https://github.com/ShawnXxy/PDFCraft"  # <-- Change this

# Personal Access Token (optional, required for private repositories)
# Leave as None or empty string for public repositories
PAT = None  # <-- Set your PAT here for private repos, e.g., "ghp_xxxx..."

# Repository type (auto-detected if None)
# Options: "github", "gitlab", "azuredevops", "bitbucket"
REPO_TYPE = None  # <-- Usually auto-detected, set manually if needed

# Branch to clone (optional, uses default branch if None)
BRANCH = None  # <-- e.g., "main", "develop"

# ============================================

def detect_repo_type(url: str) -> str:
    """Auto-detect repository type from URL."""
    url_lower = url.lower()
    if "github.com" in url_lower or "github" in url_lower:
        return "github"
    elif "gitlab.com" in url_lower or "gitlab" in url_lower:
        return "gitlab"
    elif "dev.azure.com" in url_lower or "visualstudio.com" in url_lower:
        return "azuredevops"
    elif "bitbucket.org" in url_lower or "bitbucket" in url_lower:
        return "bitbucket"
    else:
        return "github"  # Default

# Auto-detect repo type if not specified
repo_type = REPO_TYPE or detect_repo_type(REPO_URL)

print(f"Repository URL: {REPO_URL}")
print(f"Repository Type: {repo_type}")
print(f"PAT Provided: {'Yes' if PAT else 'No'}")
print(f"Branch: {BRANCH or 'default'}")

## 6. Clone Repository & Create Embeddings

Download the repository and create embeddings for all code files.

In [ ]:
from backend.data_pipeline import DatabaseManager

# Initialize the database manager
db_manager = DatabaseManager()

# Prepare the database (clone repo + create embeddings)
print(f"Processing repository: {REPO_URL}")
print("This may take a few minutes depending on the repository size...")
print("-" * 60)

documents = db_manager.prepare_database(
    repo_url_or_path=REPO_URL,
    repo_type=repo_type,
    access_token=PAT,
    branch=BRANCH
)

print("-" * 60)
print(f"✓ Repository processed successfully!")
print(f"✓ Total documents indexed: {len(documents)}")
print(f"✓ Database saved to: {db_manager.repo_paths['save_db_file']}")

## 7. Generate Wiki Structure

Use the RAG system to analyze the repository and generate a wiki structure.
This reuses the same approach as the frontend - sending a prompt to the LLM with RAG context.

In [ ]:
from backend.rag import RAG
from backend.azureai_client import AzureAIClient
from backend.config import get_model_config, get_azure_deployment_name
from adalflow.core.types import ModelType

# Initialize RAG and prepare retriever for the repository
# Use prepare_retriever which handles everything internally
rag = RAG()
rag.prepare_retriever(
    repo_url_or_path=REPO_URL,
    type=repo_type,
    access_token=PAT,
    branch=BRANCH
)

# Initialize Azure OpenAI client with proper config (not bare constructor)
config = get_model_config()
azure_client = AzureAIClient(**config.get('initialize_kwargs', {}))

print(f"✓ RAG initialized with {len(rag.transformed_docs)} documents")
print(f"✓ Azure OpenAI client ready")
print(f"✓ Deployment: {get_azure_deployment_name()}")

In [ ]:
# Import wiki structure prompt from promptstore
from backend.promptstore.wiki_structure import (
    WIKI_STRUCTURE_PROMPT,
    WIKI_STRUCTURE_CONCISE_PROMPT,
    get_language_name
)

# ============================================
# USER INPUT: Wiki configuration
# ============================================
LANGUAGE = "en"  # Language code: en, ja, zh, es, kr, etc.
COMPREHENSIVE = True  # True for full wiki (8-12 pages), False for concise (4-6 pages)

# Get the repo owner and name from URL
repo_parts = REPO_URL.rstrip('/').split('/')
owner = repo_parts[-2] if len(repo_parts) >= 2 else "unknown"
repo_name = repo_parts[-1] if len(repo_parts) >= 1 else "unknown"

# Get file tree from the database (use rag.transformed_docs from RAG object)
file_tree = "\n".join([doc.meta_data.get('file_path', '') for doc in rag.transformed_docs[:100]])
readme = "See repository README.md"  # Could be fetched if needed

# Format the prompt with actual values
wiki_prompt = (WIKI_STRUCTURE_PROMPT if COMPREHENSIVE else WIKI_STRUCTURE_CONCISE_PROMPT).format(
    owner=owner,
    repo=repo_name,
    file_tree=file_tree,
    readme=readme,
    language_name=get_language_name(LANGUAGE),
    page_count="8-12" if COMPREHENSIVE else "4-6"
)

print(f"📚 Wiki Structure Generation")
print(f"   Repository: {owner}/{repo_name}")
print(f"   Language: {get_language_name(LANGUAGE)}")
print(f"   Mode: {'Comprehensive' if COMPREHENSIVE else 'Concise'}")
print("-" * 60)

# Retrieve relevant context using RAG
print("Retrieving relevant context from repository...")
retrieved_documents = rag(wiki_prompt)

# Build context from retrieved documents
context_text = ""
if retrieved_documents and retrieved_documents[0].documents:
    docs = retrieved_documents[0].documents
    print(f"✓ Retrieved {len(docs)} relevant documents")
    
    docs_by_file = {}
    for doc in docs:
        file_path = doc.meta_data.get('file_path', 'unknown')
        if file_path not in docs_by_file:
            docs_by_file[file_path] = []
        docs_by_file[file_path].append(doc)
    
    context_parts = []
    for file_path, file_docs in docs_by_file.items():
        header = f"## File: {file_path}\n\n"
        content = "\n\n".join([d.text for d in file_docs])
        context_parts.append(f"{header}{content}")
    
    context_text = "\n\n---\n\n".join(context_parts)
    print(f"✓ Context built from {len(docs_by_file)} files")
else:
    print("⚠ No documents retrieved")

In [ ]:
# Generate wiki structure using Azure OpenAI (non-streaming for reliability)
import re

# Limit context to avoid token limits - use smaller chunk
context_truncated = context_text[:15000] if context_text else "No context available"

SYSTEM_PROMPT = f"""You are a technical documentation expert analyzing a code repository.
You have access to the following context from the repository:

{context_truncated}

Based on this context, help generate wiki documentation."""

# Build messages for the API call
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": wiki_prompt}
]

print("Generating wiki structure...")
print(f"Context size: {len(context_truncated)} chars")
print("=" * 60)

try:
    # Use non-streaming call for reliability
    response = azure_client.call(
        api_kwargs={
            "messages": messages,
            "model": get_azure_deployment_name()
        },
        model_type=ModelType.LLM
    )

    wiki_structure_response = response.choices[0].message.content
    print(wiki_structure_response)
    print("=" * 60)
    print("✓ Wiki structure generated!")

    # Parse the XML response to extract pages
    xml_match = re.search(r'<wiki_structure>[\s\S]*?</wiki_structure>', wiki_structure_response)
    if xml_match:
        print("\n📋 Parsed Wiki Pages:")
        # Extract page titles
        page_titles = re.findall(r'<page id="([^"]+)"[^>]*>[\s\S]*?<title>([^<]+)</title>', wiki_structure_response)
        for page_id, title in page_titles:
            print(f"   • {page_id}: {title}")
    else:
        print("⚠ Could not parse wiki structure XML")
        
except Exception as e:
    print(f"\n❌ Error: {type(e).__name__}")
    print(f"   Message: {e}")
    wiki_structure_response = None

## 8. Generate Wiki Page Content

Generate content for a specific wiki page using the RAG system.

In [ ]:
# Import wiki page prompt from promptstore
from backend.promptstore.wiki_page import (
    WIKI_PAGE_CONTENT_PROMPT,
    format_file_paths_list
)

# ============================================
# USER INPUT: Wiki page to generate
# ============================================
PAGE_TITLE = "System Architecture"
PAGE_FILE_PATHS = ["backend/api.py", "backend/rag.py", "backend/simple_chat.py"]

# Format the page content prompt with actual values
file_paths_list = format_file_paths_list(PAGE_FILE_PATHS, REPO_URL, BRANCH or "main")

page_prompt = WIKI_PAGE_CONTENT_PROMPT.format(
    page_title=PAGE_TITLE,
    file_paths_list=file_paths_list,
    language_name=get_language_name(LANGUAGE)
)

print(f"📄 Generating Wiki Page: {PAGE_TITLE}")
print(f"   Relevant files: {', '.join(PAGE_FILE_PATHS)}")
print(f"   Language: {get_language_name(LANGUAGE)}")
print("-" * 60)

# Retrieve context for these specific files
retrieved = rag(page_prompt)
page_context = ""
if retrieved and retrieved[0].documents:
    docs = retrieved[0].documents
    for doc in docs:
        file_path = doc.meta_data.get('file_path', 'unknown')
        page_context += f"\n## File: {file_path}\n{doc.text}\n"
    print(f"✓ Retrieved {len(docs)} relevant documents")

In [ ]:
# Generate the page content using Azure OpenAI (non-streaming)

page_context_truncated = page_context[:15000] if page_context else "No context available"

page_system_prompt = f"""You are a technical documentation expert.
You have access to the following context from the repository:

{page_context_truncated}

Based on this context, generate detailed wiki documentation."""

page_messages = [
    {"role": "system", "content": page_system_prompt},
    {"role": "user", "content": page_prompt}
]

print("Generating page content...")
print(f"Context size: {len(page_context_truncated)} chars")
print("=" * 60)

try:
    response = azure_client.call(
        api_kwargs={
            "messages": page_messages,
            "model": get_azure_deployment_name()
        },
        model_type=ModelType.LLM
    )
    
    page_content = response.choices[0].message.content
    print(page_content)
    print("=" * 60)
    print(f"✓ Page '{PAGE_TITLE}' generated!")
    print(f"   Content length: {len(page_content)} characters")
except Exception as e:
    print(f"\n❌ Error: {type(e).__name__}")
    print(f"   Message: {e}")
    page_content = None

## 9. Query the Repository (Q&A)

Ask questions about the repository using the RAG system.

In [ ]:
# Import simple chat prompt from promptstore
from backend.promptstore import SIMPLE_CHAT_SYSTEM_PROMPT

# ============================================
# USER INPUT: Ask questions about the repo
# ============================================
QUERY = "What is the main purpose of this repository and how is it structured?"

# Retrieve relevant context
print(f"❓ Query: {QUERY}")
print("-" * 60)

retrieved = rag(QUERY)
query_context = ""
if retrieved and retrieved[0].documents:
    docs = retrieved[0].documents
    for doc in docs:
        file_path = doc.meta_data.get('file_path', 'unknown')
        query_context += f"\n## File: {file_path}\n{doc.text}\n"
    print(f"✓ Retrieved {len(docs)} relevant documents")

# Truncate context to avoid token limits
query_context_truncated = query_context[:15000] if query_context else "No context available"

# Generate answer using Azure OpenAI (non-streaming)
qa_system_prompt = SIMPLE_CHAT_SYSTEM_PROMPT.format(
    repo_type=repo_type,
    repo_url=REPO_URL,
    repo_name=repo_name,
    language_name=get_language_name(LANGUAGE)
) + f"\n\nContext from repository:\n{query_context_truncated}"

qa_messages = [
    {"role": "system", "content": qa_system_prompt},
    {"role": "user", "content": QUERY}
]

print("\n📝 Answer:")
print("=" * 60)

try:
    response = azure_client.call(
        api_kwargs={
            "messages": qa_messages,
            "model": get_azure_deployment_name()
        },
        model_type=ModelType.LLM
    )
    
    answer = response.choices[0].message.content
    print(answer)
    print("=" * 60)
except Exception as e:
    print(f"\n❌ Error: {type(e).__name__}")
    print(f"   Message: {e}")
    answer = None

## View Logs

Display recent log entries.

In [ ]:
from pathlib import Path
from datetime import datetime

# Get today's log file
log_dir = Path(project_root) / "logs"
date_str = datetime.now().strftime("%y%m%d")
log_file = log_dir / f"backend-{date_str}.log"

if log_file.exists():
    with open(log_file, 'r') as f:
        lines = f.readlines()
        print(f"Last 20 log entries from {log_file.name}:")
        print("=" * 80)
        for line in lines[-20:]:
            print(line.rstrip())
else:
    print(f"Log file not found: {log_file}")